In [3]:
import pandas as pd
import numpy as np

In [ ]:
#Carga del dataframe original
df=pd.read_csv('../data/openb_pod_list_cpu0.csv')

In [ ]:
#Guardado del dataframe procesado
df.to_csv('../data/processed_data.csv', index=False)

In [2]:
#Carga el dataframe procesado
df=pd.read_csv('../data/processed_data.csv')

In [3]:
print(f"Cantidad de Columnas: {df.columns.nunique()}\nCantidad de filas: {df.shape[0]}\nCantidad de nulos: {df.isnull().sum().sum()}\nNombres:\n {df.columns}")

Cantidad de Columnas: 10
Cantidad de filas: 4334
Cantidad de nulos: 0
Nombres:
 Index(['name', 'cpu_milli', 'memory_mib', 'num_gpu', 'gpu_milli', 'qos',
       'pod_phase', 'creation_time', 'deletion_time', 'scheduled_time'],
      dtype='str')


In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.tail(5)

In [ ]:
#Eliminamos la columna que no tiene valores ya que solo aportaría ruido al modelo.
df.drop(columns=['gpu_spec'], inplace=True)

In [ ]:
#Elinamos todos los registros que tienen como estatus "Pending" ya que no aportan metricas de información para el modelo
df.dropna(subset=['scheduled_time'], inplace=True)


In [ ]:
#Eliminamos todos los registros que tienen como estatus "Failed" ya que no aportan metricas de información para el modelo
df=df[df['pod_phase']!='Failed']

In [ ]:
#       .:Feature Engineering:.

#Obtenemos la duración real del pod
df['duration_time']=df['deletion_time']-df['creation_time']

#Volvemos númericas las variables categorícas de la columna qos con el metodo del one hot encoding
df_fe=pd.get_dummies(df,columns=['qos'], drop_first=True,dtype=int)

#Eliminamos todas las columnas que no aportan valor matemático al modelo
columnas_a_eleminar=['name','pod_phase','deletion_time','creation_time','scheduled_time']
df_modelo=df_fe.drop(columns=columnas_a_eleminar)


In [10]:
print(f"\t~~~~Estadisticas del df_model~~~~\nCantidad de columnas: {df_modelo.columns.nunique()}\nCantidad de filas: {df_modelo.shape[0]}\nColumnas: {df_modelo.columns}")

	~~~~Estadisticas del df_model~~~~
Cantidad de columnas: 8
Cantidad de filas: 4334
Columnas: Index(['cpu_milli', 'memory_mib', 'num_gpu', 'gpu_milli', 'duration_time',
       'qos_Burstable', 'qos_Guaranteed', 'qos_LS'],
      dtype='str')


In [ ]:
#Guardamos el dataframe que se utilizará para entrenar el modelo
df_modelo.to_csv('../data/df_modelo.csv', index=False)

In [4]:
#Cargamos el dataframe que se utilizará para entrenar el modelo
df_modelo=pd.read_csv('../data/df_modelo.csv')

### Una vez que tenemos limpio el DataFrame, procedemos a separar los conjuntos de datos de entrenamiento y prueba para los modelos (Regresión Lineal y Red Neuronal).

In [5]:
from sklearn.model_selection import train_test_split

#Separamos las variables que fungirán como características (x) y la variable que fungirá como target (y)
X=df_modelo.drop(columns=['cpu_milli'])
y=df_modelo['cpu_milli']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [6]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False) 
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)